# Attaching a LoRA in volis, step by step

A **LoRA** is a small file of adjustments for one particular model. volis never changes a model's
own files: it loads the original model and applies the LoRA on top. In volis, you never attach a
LoRA in code. You **put it in its own folder and name the model it belongs to**, and it shows up in
the window as a model of its own.

This notebook goes through that once, with real volis code, so you can see each step happen:

1. Set up, and see which models are installed.
2. The one rule: a LoRA fits only the model it was made for.
3. **Translator LoRA:** make a practice LoRA, install it, check that volis finds it, and compare
   translations with and without it.
4. **Recognizer LoRA:** the same steps for speech recognition (needs a LoRA you've downloaded).
5. Turning a translator LoRA from training into the one-file format volis needs.
6. Removing the practice LoRA again.

Run the cells in order, top to bottom, with **Shift+Enter**.

### How to open this notebook

**In VS Code (easiest):** open the volis folder, open this file, click **Select Kernel** at the top
right, then **Python Environments → .venv**. If VS Code offers to install `ipykernel`, say yes.
That adds one small package to volis's environment, and running `setup.ps1` later removes it
again.

**Or with Jupyter in a browser,** from the volis folder in PowerShell:

```powershell
uv run --with jupyterlab jupyter lab notebooks\attach-a-lora.ipynb
```

## Step 1: set up, and see what's installed

Two things happen first, exactly as when volis starts:
- find the volis folder (one level above this notebook);
- switch on volis's **offline settings** *before* importing anything else. That keeps every
  library from reaching the internet, and every cache inside the volis folder.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from volis import paths
paths.apply_offline_environment(ROOT)   # before anything else from volis, like volis itself does

from volis import models, gguf
from volis.models import Role, Failed

MT = paths.mt_dir(ROOT)
ASR = paths.asr_dir(ROOT)
print("volis folder:      ", ROOT)
print("translators folder:", MT)
print("recognizers folder:", ASR)

In [ ]:
def show_translators():
    print("TRANSLATORS")
    for t in models.discover_translators(MT):
        if isinstance(t, Failed):
            print(f"  [x] {t.dir_name}: {t.error}")
            continue
        status = "ok" if t.enabled() else f"can't be used: {t.unusable or 'missing ' + ', '.join(t.missing)}"
        lora = f"   + LoRA {t.lora.name} (scale {t.lora_scale})" if t.lora else ""
        print(f"  {'[+]' if t.enabled() else '[x]'} {t.id}   ({t.backend}){lora}\n        {status}")

def show_recognizers():
    print("RECOGNIZERS")
    for e in models.discover(ASR, Role.ASR):
        if isinstance(e, Failed):
            print(f"  [x] {e.dir_name}: {e.error}")
            continue
        status = "ok" if e.enabled() else f"can't be used: {e.unusable or 'missing ' + ', '.join(e.missing_files())}"
        print(f"  {'[+]' if e.enabled() else '[x]'} {e.dir_name}   ({e.backend}) {e.detail}\n        {status}")

show_translators()
print()
show_recognizers()

## Step 2: the one rule

**A LoRA only fits the exact model it was made for,** like a key cut for one lock. A LoRA made for
Gemma 4 E2B won't fit Gemma 4 E4B, even though both are Gemma 4.

So before installing any LoRA, look at what it says about itself:
- a **translator LoRA** is one `.gguf` file. Its own description says it's an adapter, and which
  kind of model it's for (`general.architecture`);
- a **recognizer LoRA** is a folder with `adapter_config.json`. Its `base_model_name_or_path` line
  names the exact base model.

This cell reads either kind. Point `LORA_TO_CHECK` at a file or folder you've downloaded, or leave
it empty for now: Step 3 makes a practice one to look at.

In [ ]:
import json

def describe_lora(path):
    path = Path(path)
    if path.suffix == ".gguf":
        meta = gguf.read_metadata(path)
        kind = meta.get("general.type", "model")
        print(f"{path.name}")
        print(f"  this file is a:       {kind}  {'(a LoRA adapter)' if kind == 'adapter' else '(a full model, not a LoRA)'}")
        print(f"  made for models of:   {meta.get('general.architecture')}")
        print(f"  its name:             {meta.get('general.name')}")
    elif (path / "adapter_config.json").is_file():
        cfg = json.loads((path / "adapter_config.json").read_text(encoding="utf-8"))
        print(f"{path.name}  (a recognizer-style LoRA folder)")
        print(f"  made for exactly:     {cfg.get('base_model_name_or_path')}")
        print(f"  LoRA type, size:      {cfg.get('peft_type')}, r = {cfg.get('r')}")
    else:
        print(f"{path}: not a .gguf file, and no adapter_config.json in it")

LORA_TO_CHECK = ""   # e.g. r"D:\Downloads\some-lora"
if LORA_TO_CHECK:
    describe_lora(LORA_TO_CHECK)
else:
    print("Nothing to check yet: Step 3 makes a practice LoRA.")

## Step 3: a translator LoRA

### 3a. Choose the translator to attach it to

The LoRA goes on **one** installed translator. It must be a GGUF translator. Change
`BASE_CONTAINS` to choose a different one from the list in Step 1.

In [ ]:
BASE_CONTAINS = "gemma-4-E4B"     # part of the translator's name from Step 1

candidates = [t for t in models.discover_translators(MT)
              if not isinstance(t, Failed) and t.backend == "llamacpp" and t.lora is None
              and BASE_CONTAINS.lower() in t.id.lower() and t.enabled()]
if not candidates:
    raise SystemExit(f"No usable GGUF translator with '{BASE_CONTAINS}' in its name. Pick one from Step 1.")
base = candidates[0]
print("The LoRA will be attached to:", base.id)
print("  file:", base.path)
print("  kind of model:", base.architecture)

### 3b. Make a practice LoRA

You don't have a trained translator LoRA yet, so this makes a **practice** one, with volis's own
test tool (`scripts\make_test_lora.py`). It nudges a tiny part of the model at random. It teaches
the model nothing useful, but it's a real LoRA file, made for exactly this model, so every step
after this one works just like it would with a real LoRA.

`STRENGTH` is how big the nudge is. With 0, the translations must come out identical to the model
without the LoRA, which proves it attached and does nothing. With 0.05, they should differ a
little, which proves it's really being applied.

**With a real LoRA,** skip this cell and set `ADAPTER_FILE` to your `.gguf` file instead.

In [ ]:
import subprocess, tempfile

STRENGTH = 0.05
ADAPTER_FILE = Path(tempfile.gettempdir()) / "volis-practice-lora.gguf"

result = subprocess.run(
    [sys.executable, str(ROOT / "scripts" / "make_test_lora.py"), str(base.path), str(ADAPTER_FILE),
     "--strength", str(STRENGTH)],
    capture_output=True, text=True, encoding="utf-8", errors="replace")
print(result.stdout or result.stderr)
describe_lora(ADAPTER_FILE)

### 3c. Install it

Installing a translator LoRA is two things:

1. **Its own folder** in `models\mt\`, holding the LoRA file.
2. **A small settings file**, `volis-python.toml`, in that folder, saying:
   - `base`: which translator it goes on, exactly as Step 1 listed it;
   - `scale`: how strongly to apply it (1.0 is normal; 0.5 is half as strong);
   - `name`: what to call it in the window (optional).

That's all. No code, no changes to the original model. This cell does both, and refuses to
overwrite a folder that's already there.

In [ ]:
import shutil

LORA_FOLDER_NAME = "practice-lora"      # the new folder's name in models\mt\
SCALE = 1.0

folder = MT / LORA_FOLDER_NAME
if folder.exists():
    raise SystemExit(f"{folder} already exists. Choose another name, or remove it first (Step 6).")
folder.mkdir()
shutil.copy2(ADAPTER_FILE, folder / ADAPTER_FILE.name)
(folder / "volis-python.toml").write_text(
    f'base = "{base.id}"\n'
    f"scale = {SCALE}\n"
    f'name = "Practice LoRA"\n',
    encoding="utf-8")

print("Made", folder)
for p in sorted(folder.iterdir()):
    print("  ", p.name)
print("\nvolis-python.toml says:\n" + (folder / "volis-python.toml").read_text(encoding="utf-8"))

### 3d. Check that volis finds it

Now run volis's model discovery again: the same thing **Rescan** in the window, or `--report`, does.
The LoRA should appear as a translator of its own, joined to its base, and marked ok. If anything's
wrong (the base isn't installed, or the LoRA was made for another kind of model), the line says
exactly what.

In [ ]:
show_translators()
lora_entry = next(t for t in models.discover_translators(MT)
                  if not isinstance(t, Failed) and t.lora is not None and t.lora.parent == folder)
print("\nThe LoRA entry:")
print("  id in volis:  ", lora_entry.id)
print("  shown as:     ", lora_entry.name)
print("  base model:   ", lora_entry.path)
print("  LoRA file:    ", lora_entry.lora)
print("  usable:       ", lora_entry.enabled(), lora_entry.unusable)

### 3e. Compare: with and without the LoRA

The real test of a LoRA: the same sentences through the plain model and through the model with
the LoRA, side by side. This uses volis's own translator code and prompt, exactly as the app does.

The two models are loaded **one at a time**, and each is closed before the next, so they don't both
need graphics memory at once. Expect a short wait for each.

In [ ]:
from volis import translate
from volis.translate import prompts, TranslationRequest

SOURCE, TARGET = "en", "ar"
SENTENCES = [
    "Where is the nearest pharmacy?",
    "My phone died, can I use your computer to check my email?",
    "We'll meet at the market at four o'clock.",
]

prompt = prompts.load(paths.prompts_dir(ROOT))

def translate_all(entry):
    t = translate.load(entry, prompt, "auto")
    try:
        return [t.translate(TranslationRequest(s, SOURCE, TARGET)).text for s in SENTENCES], t.device
    finally:
        t.close()

plain, dev1 = translate_all(base)
with_lora, dev2 = translate_all(lora_entry)

print(f"{SOURCE} -> {TARGET}   (plain on {dev1}, with LoRA on {dev2})\n")
for s, a, b in zip(SENTENCES, plain, with_lora):
    print(s)
    print("   plain:     ", a)
    print("   with LoRA: ", b, "" if a != b else "   (same)")
    print()
same = sum(a == b for a, b in zip(plain, with_lora))
print(f"{same} of {len(SENTENCES)} came out identical.")
print("With STRENGTH = 0 all should be identical; with 0.05 some should differ.")

### What you just did, in the window

Open volis and press **Rescan** (or restart it): the translator list now has *Practice LoRA*.
Choosing it is the same as choosing any other translator. A real LoRA you install this way works
exactly the same.

## Step 4: a recognizer LoRA

A recognizer LoRA comes as a **folder** with two files: `adapter_config.json` and
`adapter_model.safetensors`. Installing it:

1. Copy that folder into `models\asr\`.
2. Make sure the base model it was made for is in `models\asr\` too. volis reads the base's name
   from `adapter_config.json` (`openai/whisper-small` → looks for a folder called `whisper-small`).
3. Only if your base folder has a different name, add `volis-python.toml` with
   `base = "<that folder's name>"`.

There's no practice recognizer LoRA, so this part needs one you've downloaded. Set `ASR_LORA_FOLDER`
to it, and `TEST_WAV` to a short recording (WAV, MP3 or M4A) in its language. Leave them empty to skip.

In [ ]:
ASR_LORA_FOLDER = ""      # e.g. r"D:\Downloads\whisper-small-algerian-lora"
TEST_WAV = ""             # e.g. r"D:\recordings\arabic-test.wav"
LANGUAGE = "ar"

if not ASR_LORA_FOLDER:
    print("Skipped: set ASR_LORA_FOLDER to a downloaded recognizer LoRA folder to try this part.")
else:
    src = Path(ASR_LORA_FOLDER)
    describe_lora(src)
    wanted_base = json.loads((src / "adapter_config.json").read_text(encoding="utf-8"))["base_model_name_or_path"]
    base_folder = ASR / wanted_base.rstrip("/").split("/")[-1]
    print("\nvolis will look for its base in:", base_folder, "->", "found" if base_folder.is_dir() else "NOT FOUND")
    dest = ASR / src.name
    if dest.exists():
        print(f"{dest} already exists: not copying again.")
    else:
        shutil.copytree(src, dest)
        print("Copied to", dest)
    print()
    show_recognizers()

In [ ]:
if ASR_LORA_FOLDER and TEST_WAV:
    from volis import asr
    from volis.filesource import read_16k_mono   # volis's own reader: any audio file, to 16 kHz mono

    audio = read_16k_mono(Path(TEST_WAV))

    engines = {e.dir_name: e for e in models.discover(ASR, Role.ASR) if not isinstance(e, Failed)}
    lora_engine = engines[Path(ASR_LORA_FOLDER).name]
    base_engine = engines[Path(lora_engine.settings["base_dir"]).name]

    for label, engine in (("plain", base_engine), ("with LoRA", lora_engine)):
        model = asr.load(engine)
        try:
            print(f"{label:10}", model.transcribe(audio, LANGUAGE).text)
        finally:
            model.close()
else:
    print("Skipped: set ASR_LORA_FOLDER and TEST_WAV above to compare.")

## Step 5: a translator LoRA from training needs one conversion

Training (with `peft`, Unsloth and the rest) produces a LoRA **folder**: `adapter_config.json` plus
`adapter_model.safetensors`. That's fine for recognizers, as in Step 4. But volis's translators are
GGUF files, so a translator LoRA has to become **one `.gguf` file** first.

llama.cpp has a script for this, `convert_lora_to_gguf.py`. A copy is already on this computer,
in model-converter. It needs the LoRA folder, and the **original Hugging Face version** of the base
model (its `config.json`), and it runs in model-converter's environment, which has the libraries it
needs. This cell only prints the command to run. It doesn't run anything.

In [ ]:
LORA_FROM_TRAINING = r"D:\path\to\my-lora"                      # adapter_config.json + adapter_model.safetensors
HF_BASE_MODEL      = r"D:\path\to\gemma-4-E4B-it (Hugging Face)"  # the original model folder, with config.json
OUT_FILE           = r"D:\path\to\my-lora.gguf"

script = Path(r"D:\AI_Data\projects\model-converter\qwen-to-gguf\llama.cpp\convert_lora_to_gguf.py")
print("Run this from D:\\AI_Data\\projects\\model-converter, in PowerShell:\n")
print(f'uv run python "{script}" "{LORA_FROM_TRAINING}" --base "{HF_BASE_MODEL}" --outfile "{OUT_FILE}" --outtype f16')
print("\nThen install OUT_FILE exactly as in Step 3c.")

## Step 6: remove the practice LoRA

Removing a LoRA is just deleting its folder. The base model is never touched. This cell removes the
practice one made above, and its temporary file. It only runs if you set `REMOVE = True`.

In [ ]:
REMOVE = False

if REMOVE:
    shutil.rmtree(MT / LORA_FOLDER_NAME, ignore_errors=True)
    ADAPTER_FILE.unlink(missing_ok=True)
    print("Removed. Translators now:\n")
    show_translators()
else:
    print("Set REMOVE = True and run this cell again to remove the practice LoRA.")